# GMV 추종 + 괴리율 리밸런싱 백테스트 — v2 (괴리율 범위 확장 + t+1 체결)
`gmv_backtest.ipynb`(v1)에서 `양방향_전략_보완점.md`의 1·5번 항목을 반영한 버전. 결과는 `backtest_v2/`에 저장 (v1 결과는 그대로 둠)

**v1 → v2 변경점**
| 항목 | v1 | v2 |
|---|---|---|
| 괴리율 후보 | 1% ~ 10% (10개) | 1% ~ 10%, 12·15·20·25·30%, **리밸런싱 없음(∞)** (16개) |
| 체결 시점 | 괴리율을 계산한 **그날 종가**에 바로 체결 | **다음 거래일 종가**에 체결 (`EXEC_LAG = 1`) |
| 체결 비중 | 그날 장 마감 후 계산한 GMV | **신호를 낸 날** 계산한 GMV (체결일엔 새로 계산하지 않음) |
| 첫 매수 | 시작일에 그날 GMV로 매수 | 시작일 **전날** 계산한 GMV를 시작일 종가에 매수 |
| 휴장일 | 쉬는 시장도 직전 가격으로 매매 가능 | 매매할 종목이 있는 시장이 **하나라도 쉬면 체결을 미룸** → 모두 열린 첫날 체결 |

- "리밸런싱 없음"도 시나리오 순위에 넣어서, 괴리율 규칙이 그냥 들고 있는 것보다 나은지 같이 비교
- 14장에서 같은 시나리오를 그날 체결(v1 방식)로도 돌려 **t+1 체결의 영향**만 따로 확인

`금프2 프로젝트 회의.md`의 "전략 단순화" 구현

**전략 요약**
1. **가상 포트폴리오**: 매일 롤링 윈도우(3개월 / 6개월)로 공분산을 추정해 **GMV(최소분산) 포트폴리오**를 항상 유지 (거래비용 없음, 기준선 역할)
2. **실제 포트폴리오**: 투자 시작일에 가상 포트폴리오와 똑같이 맞춘 뒤, 매매하지 않고 보유(가격 변동에 따라 비중이 흘러감)
3. 매일 장 마감 후 **괴리율 = |가상 누적수익률 − 실제 누적수익률|** 을 계산해서, 괴리율이 임계값 이상이면 **다음 거래일 종가**에 GMV 비중으로 리밸런싱 (거래비용 발생)
4. 시나리오 = 윈도우 2개 × 괴리율 16개 = **32개**
5. **2024-12-31까지(in-sample)** 성과로 최적 조합 선택 → **2025-01-01 ~ 최신(out-of-sample)** 성과 측정

**회의록에 정해지지 않아서 이 코드에서 정한 것들** (아래 설정 셀에서 바꿀 수 있음)

| 항목 | 이 코드의 선택 | 이유 |
|---|---|---|
| 현금(무위험자산)의 역할 | 샤프·소르티노 계산 시 무위험 수익률 = 0으로만 사용, **GMV 계산에는 넣지 않음** | 표준편차 0인 현금을 넣으면 GMV는 항상 "현금 100%"가 되어 전략이 성립하지 않음 |
| 괴리율 기준점 | **마지막 리밸런싱 시점부터**의 누적수익률 차이 (리밸런싱하면 0으로 초기화) | 투자 시작일 기준으로만 재면, 한 번 벌어진 차이는 리밸런싱해도 없어지지 않아 그 뒤로 매일 리밸런싱하게 됨 |
| 공매도 | 불가 (비중 ≥ 0, 합 = 1) | 실제 서버에서 현물 매수만 가능하다고 가정 |
| 공분산 추정 | 표본 공분산 + Ledoit-Wolf 수축(상관계수만 수축, 각 자산 분산은 그대로) | 종목 수(최대 약 230) > 관측 수(63·126일)라서 표본 공분산이 특이행렬이 됨 |
| 통화 | 모든 자산을 **원화(KRW)** 로 환산 (USD/KRW: FRED DEXKOUS), 시작 자금은 원화 현금 | 코스피·나스닥·코인을 한 포트폴리오로 묶으려면 통화 통일 필요 |
| 거래비용 | `trading_costs.md` (서버 규칙) 그대로: **매매비용 + 환전비용** | 아래 표 참고 |
| 가격 | 모든 자산 **배당·분배금 미포함 가격**(분할만 조정) | 서버는 분배금(이자) 없이 가격 변동만 반영 |
| 투자금 | **1억 원**, 주식·ETF는 **정수 주**만 매매 (코인은 소수점 매매), 못 산 돈은 원화 현금(수익 0) | 실제 1억 계좌에서 가능한 포트폴리오로 시뮬레이션 |
| 무기한 선물 | 제외 (현물만 사용) | 기초자산이 현물과 같아 공분산이 거의 겹치고, 증거금·펀딩비 모델링이 별도로 필요 |
| 롤링 윈도우 | 3개월 = 63거래일, 6개월 = 126거래일 | |
| 최적 조합 선정 | CAGR·샤프·소르티노·칼마·MDD 5개 지표 순위의 **평균 순위**가 가장 좋은 조합 | "여러 지표 종합" |

In [ ]:
# =============================================================================
# 0. 설정
# =============================================================================
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import yfinance as yf

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_DIR = Path("/Users/kasia/Desktop/학교/26-2/금프2/프로젝트")
MD_PATH = PROJECT_DIR / "tradable_assets.md"
OUT_DIR = PROJECT_DIR / "backtest_v2"       # 결과물 저장 폴더 (v1은 backtest/)
OUT_DIR.mkdir(exist_ok=True)

DATA_START = "1996-01-01"                   # 회의록: 1996-01-01 ~ 최신
IS_END = pd.Timestamp("2024-12-31")         # in-sample 마지막 날
OOS_START = pd.Timestamp("2025-01-01")      # out-of-sample 시작

WINDOWS = {"3M": 63, "6M": 126}             # 롤링 윈도우 (거래일 수)
NO_REBAL = np.inf                           # 괴리율 ∞ = 첫 매수 후 리밸런싱 안 함
THRESHOLDS = [i / 100 for i in range(1, 11)] + [0.12, 0.15, 0.20, 0.25, 0.30] + [NO_REBAL]   # 괴리율 후보 16개
EXEC_LAG = 1                                # 체결 지연 (거래일): 1 = 신호 다음 날 종가 체결, 0 = 그날 종가 체결(v1)

def tlabel(t):
    """괴리율 표시용 문자열 (∞ → '없음')"""
    return "없음" if np.isinf(t) else f"{t:.0%}"

BASE_CCY = "KRW"            # "KRW": 전부 원화 환산 / "LOCAL": 환산 없이 각 통화 수익률 그대로
COV_METHOD = "ledoit_wolf"  # "ledoit_wolf" 또는 "sample"
# --- 투자금 ---
CAPITAL = 100_000_000                         # 1억 원 (원화 현금으로 시작)
INTEGER_SHARES = {"KOSPI": True, "NASDAQ": True, "CRYPTO": False}   # True = 정수 주만 매매 가능

# --- 거래비용: trading_costs.md (서버 get_rules 기준, 2026-11-01) ---
# (1) 현물 매매비용: 매수·매도 각각 부과, 수수료 + 슬리피지
COST = {
    "KOSPI":  0.0015 + 0.0005,   # 0.20%  (왕복 약 0.40%)
    "NASDAQ": 0.0005 + 0.0003,   # 0.08%  (왕복 약 0.16%)
    "CRYPTO": 0.0010 + 0.0002,   # 0.12%  (왕복 약 0.24%)
}
# (2) 환전 스프레드: 양방향 모두 부과 (환전 금액 대비)
#     시장별 통화 주머니: 코스피=KRW, 나스닥=USD, 코인=USDT
MARKET_CCY = {"KOSPI": "KRW", "NASDAQ": "USD", "CRYPTO": "USDT"}
FX_COST = {
    frozenset({"KRW", "USD"}):  0.0090,   # 원화↔달러 0.90%
    frozenset({"USD", "USDT"}): 0.0015,   # 달러↔USDT 0.15%
    frozenset({"KRW", "USDT"}): 0.0105,   # 원화↔USDT 1.05%
}
ANN = 252                   # 연율화 계수 (거래일)

# 차트 공통 스타일 (한글 폰트 + 시각화 팔레트)
plt.rcParams.update({
    "font.family": "AppleGothic", "axes.unicode_minus": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e8e7e2", "grid.linewidth": 0.8,
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"     # 범주형 1~3번 슬롯
TEXT_2 = "#52514e"
SEQ_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

## 1. 데이터 불러오기
앞에서 만든 CSV를 그대로 사용:
- 코스피: DataGuide **수정주가** (`kospi_adj_close_30y.csv`, 분할 조정·배당 미포함). DataGuide에 없는 국채 ETF 3개 + 맥쿼리인프라는 yfinance(`.KS`) **Close**로 보충
- 나스닥: yfinance **Close** (`nasdaq_close_30y.csv`, 분할 조정·배당 미포함)
- 코인: 바이낸스 현물 종가 (`crypto_spot_close.csv`)
- 환율: FRED `DEXKOUS` (원/달러)

⚠️ 수익률 계산에는 원주가가 아닌 **수정주가**를 써야 함 (원주가로 계산하면 액면분할일에 가짜 급락이 생김)
⚠️ 서버 규칙("분배금(이자)은 없고 가격 변동만 반영")에 맞춰 **모든 자산을 배당·분배금 없는 가격**으로 통일
   → 야후 Adj Close(배당 포함)는 쓰지 않음. 특히 국채 ETF는 수익 대부분이 이자라서 가격만 보면 수익률이 거의 0에 가까움

In [ ]:
# --- 1-1. md 파일에서 종목명 읽기 (차트·결과표에 이름 표시용) ---
md_text = MD_PATH.read_text(encoding="utf-8")

def md_section(title):
    """md 파일에서 '## {title}'로 시작하는 섹션 본문만 잘라 반환"""
    return re.search(rf"^## {title}.*?$(.*?)(?=^## |\Z)", md_text, flags=re.M | re.S).group(1)

kospi_names = {m.group(1): m.group(2).strip()
               for m in re.finditer(r"(\d{6})\s+([^/\n]+)", md_section("코스피"))}
nasdaq_names = {m.group(1): m.group(2).strip()
                for m in re.finditer(r"\b([A-Z][A-Z.]{0,5})\s+([^/\n]+)", md_section("나스닥"))}
crypto_names = {m.group(1): m.group(2).strip()
                for m in re.finditer(r"\b([A-Z0-9]+USDT)\s+([^/\n]+)", md_section("코인 현물"))}
print(f"md 종목 수 - 코스피 {len(kospi_names)}, 나스닥 {len(nasdaq_names)}, 코인 현물 {len(crypto_names)}")

In [ ]:
# --- 1-2. 가격 CSV 읽기 ---
kospi = pd.read_csv(PROJECT_DIR / "kospi_adj_close_30y.csv", index_col=0, parse_dates=True)
kospi.columns = [c.lstrip("A") for c in kospi.columns]          # 'A005930' → '005930'
KOSPI_LAST = kospi.index.max()          # DataGuide 마지막 날짜 (yfinance 보충분이 더 길어도 이 날짜 기준)
KOSPI_DAYS = kospi.index                # 코스피 개장일 = DataGuide에 있는 날짜 (보충 ETF 합치기 전)

# DataGuide(일반기업 보통주)에 없는 코스피 종목 → yfinance로 보충 (Close = 분배금 미포함 가격)
missing_ks = [c for c in kospi_names if c not in kospi.columns]
yf_ks = yf.download([f"{c}.KS" for c in missing_ks], start=DATA_START,
                    auto_adjust=False, progress=False)["Close"]
yf_ks.columns = [c.replace(".KS", "") for c in yf_ks.columns]
yf_ks.to_csv(PROJECT_DIR / "kospi_etf_close.csv")           # 재현용으로 저장
kospi = kospi.join(yf_ks, how="outer")
print("yfinance로 보충한 코스피 종목:", {c: kospi_names[c] for c in missing_ks})

nasdaq = pd.read_csv(PROJECT_DIR / "nasdaq_close_30y.csv", index_col=0, parse_dates=True)     # 분할만 조정된 가격
crypto = pd.read_csv(PROJECT_DIR / "crypto_spot_close.csv", index_col=0, parse_dates=True)

# --- 1-3. 원/달러 환율 (FRED) ---
fx = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXKOUS",
                 index_col=0, parse_dates=True).iloc[:, 0]
fx = pd.to_numeric(fx, errors="coerce").dropna()                 # 미국 공휴일은 빈칸 → 제거
fx.to_csv(PROJECT_DIR / "fx_usdkrw.csv")

for name, df in [("코스피", kospi), ("나스닥", nasdaq), ("코인", crypto)]:
    print(f"{name}: {df.index.min().date()} ~ {df.index.max().date()}, {df.shape[1]}종목")
print(f"USD/KRW: {fx.index.min().date()} ~ {fx.index.max().date()}")

## 2. 하나의 가격 패널로 합치기
- **거래일 달력** = 코스피 거래일 ∪ 나스닥 거래일 (평일). 코인은 그 날짜의 종가만 사용 → 주말 수익률은 월요일에 반영
- **마지막 날짜** = 세 시장 모두 데이터가 있는 마지막 날 (회의록: "모든 종목이 있는 값만")
- 한쪽 시장만 쉬는 날(설·추석 연휴 포함)은 직전 가격으로 채움 (수익률 0)
  - 단 **각 자산의 마지막 데이터 날짜 이후는 채우지 않음** → 상장폐지·거래중단 자산은 그 뒤로 투자 대상에서 빠짐
  - (일수 제한으로 채우면 2017년 추석처럼 평일 6일 연휴 때 코스피 전 종목이 반년간 빠지는 문제가 생김)
- 장 마감 시각 차이(코스피 15:30 KST, 나스닥 16:00 ET, 코인 00:00 UTC)는 무시하고 같은 날짜로 취급
- **개장일 표시**: 코스피 = DataGuide에 날짜가 있는 날, 나스닥 = yfinance에 날짜가 있는 날, 코인 = 매일.
  직전 가격으로 채운 날(휴장일)은 가치 평가에만 쓰고 **그 시장 종목은 매매하지 않음** (5장)

In [ ]:
END = min(KOSPI_LAST, nasdaq.index.max(), crypto.index.max())   # 세 시장 공통 마지막 날
cal = kospi.index.union(nasdaq.index)
cal = cal[(cal >= DATA_START) & (cal <= END)]

def align(df):
    """거래일 달력에 맞추고, 휴장일은 직전 가격으로 채움 (단 자산별 마지막 데이터 날짜까지만)"""
    full = df.reindex(df.index.union(cal)).sort_index()
    last = full.apply(lambda s: s.last_valid_index())            # 자산별 마지막 데이터 날짜
    filled = full.ffill()
    after_last = full.index.to_numpy()[:, None] > last.to_numpy()[None, :]
    return filled.mask(after_last).reindex(cal)

# 자산 ID에 시장 접두어를 붙여 구분 (예: KS:005930, NQ:AAPL, CR:BTCUSDT)
px_local = pd.concat({
    "KS": align(kospi), "NQ": align(nasdaq), "CR": align(crypto),
}, axis=1)
px_local.columns = [f"{m}:{c}" for m, c in px_local.columns]

# 자산 정보표: 이름, 시장, 통화, 거래비용
def asset_info(aid):
    m, c = aid.split(":")
    if m == "KS": return kospi_names.get(c, c), "KOSPI", "KRW"
    if m == "NQ": return nasdaq_names.get(c, c), "NASDAQ", "USD"
    return crypto_names.get(c, c), "CRYPTO", "USD"            # USDT ≈ USD로 취급

meta = pd.DataFrame([asset_info(a) for a in px_local.columns],
                    index=px_local.columns, columns=["name", "market", "ccy"])

# 통화 환산: USD 자산 × (원/달러)
if BASE_CCY == "KRW":
    fx_cal = fx.reindex(fx.index.union(cal)).ffill().reindex(cal)
    usd_cols = meta.index[meta.ccy == "USD"]
    px = px_local.copy()
    px[usd_cols] = px_local[usd_cols].mul(fx_cal, axis=0)
else:
    px = px_local.copy()

# --- 1주 가격(원주가, 원화) 패널: 1억으로 '몇 주' 살 수 있는지 계산용 ---
# 수익률은 수정주가로, 주식 수는 그날의 실제 가격(원주가)으로 계산해야 함
#  (예: 삼성전자 2018년 분할 전 1주 = 265만 원 → 수정주가 5.3만 원으로 계산하면 50배 많이 산 것으로 잘못 계산됨)
kospi_raw = pd.read_csv(PROJECT_DIR / "kospi_close_30y.csv", index_col=0, parse_dates=True)
kospi_raw.columns = [c.lstrip("A") for c in kospi_raw.columns]
kospi_raw = kospi_raw.join(yf_ks, how="outer")              # 보충 ETF는 분할 이력이 없어 Close = 원주가
nasdaq_raw = pd.read_csv(PROJECT_DIR / "nasdaq_raw_close_30y.csv", index_col=0, parse_dates=True)
px_raw = pd.concat({"KS": align(kospi_raw), "NQ": align(nasdaq_raw), "CR": align(crypto)}, axis=1)
px_raw.columns = [f"{m}:{c}" for m, c in px_raw.columns]
px_raw = px_raw[px_local.columns]
if BASE_CCY == "KRW":
    px_raw[usd_cols] = px_raw[usd_cols].mul(fx_cal, axis=0)

# 일간 수익률: 데이터가 없는 날은 NaN 유지 (0으로 채우지 않음)
rets = px.pct_change(fill_method=None)
rets = rets.where(px.notna() & px.shift(1).notna())

assets = px.columns.to_numpy()
R = rets.to_numpy()                                   # (거래일 × 자산) numpy 배열 — 속도용
cost_vec = meta["market"].map(COST).to_numpy()        # 자산별 편도 매매비용
CCYS = ["KRW", "USD", "USDT"]
ccy_onehot = np.array([(meta["market"].map(MARKET_CCY) == c).to_numpy() for c in CCYS], dtype=float)  # (3 × 자산)
UNIT_PX = px_raw.to_numpy()                           # 1주 가격 (원)
INT_MASK = meta["market"].map(INTEGER_SHARES).to_numpy(dtype=bool)   # 정수 주 제약이 있는 자산

# --- 시장별 개장일 (거래일 × 시장) → 자산별 개장 여부 (거래일 × 자산) ---
MKT_OPEN = pd.DataFrame({
    "KOSPI": cal.isin(KOSPI_DAYS),
    "NASDAQ": cal.isin(nasdaq.index),
    "CRYPTO": np.ones(len(cal), dtype=bool),          # 코인은 365일 거래
}, index=cal)
ASSET_OPEN = MKT_OPEN[meta["market"]].to_numpy()      # (거래일 × 자산) True = 그날 매매 가능
print("달력 중 휴장일:", {m: int((~MKT_OPEN[m]).sum()) for m in ["KOSPI", "NASDAQ"]}, "일")
print(f"패널: {cal[0].date()} ~ {cal[-1].date()}, {len(cal)}거래일 × {len(assets)}자산 (기준통화 {BASE_CCY})")

In [ ]:
# 데이터 품질 점검: 일간 ±50% 넘는 수익률 (원화 환산 기준)
big = rets.stack()
big = big[big.abs() > 0.5]
print("일간 |수익률| > 50%:", len(big), "건")
big.rename("ret").reset_index().assign(
    name=lambda d: d["level_1"].map(meta["name"])).sort_values("ret").head(10)

## 3. 공분산 추정과 GMV 계산

### 3-1. 공분산: Ledoit-Wolf 수축
윈도우 길이(63일·126일)보다 종목 수(최대 약 230개)가 많아서 표본 공분산은 **특이행렬**(역행렬이 없음)이 됨
→ GMV가 우연히 상쇄되는 조합에 몰려 표본 밖에서는 엉망이 됨.

해결: **상관계수 행렬만** 단위행렬 쪽으로 수축(Ledoit-Wolf 최적 강도)하고, 각 자산의 분산은 표본값 그대로 유지.
(분산까지 평균 쪽으로 수축하면 국채 ETF처럼 변동성 작은 자산과 코인처럼 큰 자산의 차이가 흐려져 GMV가 왜곡됨)

In [ ]:
def estimate_cov(Rw):
    """
    Rw: (T일 × N자산) 수익률 행렬 (NaN 없음)
    반환: (N × N) 공분산 행렬, 수축 강도
    """
    T, N = Rw.shape
    X = Rw - Rw.mean(axis=0)
    sd = X.std(axis=0)                                  # 각 자산 표준편차 (그대로 보존)

    if COV_METHOD == "sample":
        S = X.T @ X / T
        return S + np.eye(N) * 1e-10 * np.trace(S) / N, 0.0   # 아주 작은 릿지 (수치 안정용)

    # --- 표준화 수익률에 Ledoit-Wolf 적용 (목표 = 단위행렬, 즉 상관 0) ---
    Z = X / sd
    C = Z.T @ Z / T                                     # 표본 상관행렬 (대각 = 1)
    delta = ((C - np.eye(N)) ** 2).sum() / N            # 표본 행렬과 목표 행렬의 거리
    Z2 = Z ** 2
    beta = ((Z2.T @ Z2) / T - C ** 2).sum() / (N * T)   # 표본 상관행렬의 추정 오차
    shrink = 0.0 if delta == 0 else min(beta, delta) / delta   # 최적 수축 강도 (0~1)
    C_s = shrink * np.eye(N) + (1 - shrink) * C
    return sd[:, None] * C_s * sd[None, :], shrink

### 3-2. GMV (롱온리) 풀이: Active-set 방법
문제: $\min_w \; w^\top \Sigma w \quad \text{s.t.} \; \sum w_i = 1,\; w_i \ge 0$

매일 × 2개 윈도우 × 약 7,700일 = 1만 5천 번 넘게 풀어야 해서 범용 최적화(SLSQP)는 너무 느림.
롱온리 GMV는 "비중 0으로 묶인 자산 집합"만 맞히면 나머지는 닫힌 해 $w_F \propto \Sigma_{FF}^{-1}\mathbf{1}$ 이므로
**primal active-set 방법**으로 정확히 풂. 전날 해를 시작점으로 쓰면(warm start) 보통 몇 번 만에 끝남.

최적 조건(KKT): 편 자산 $F$는 $(\Sigma w)_i = \lambda$, 0인 자산은 $(\Sigma w)_i \ge \lambda$
(= 그 자산을 조금 사면 오히려 분산이 늘어남)

In [ ]:
def gmv_long_only(S, w0, tol=1e-12, max_iter=2000):
    """
    S : (N × N) 공분산 행렬 (양의 정부호)
    w0: 시작점 (w0 ≥ 0, 합 1). 0인 자산은 '비중 0 고정' 집합에서 시작
    반환: 최적 비중 w, 반복 횟수
    """
    scale = np.median(np.diag(S))
    S = S / scale                                    # 수치 안정용 스케일 (해는 동일)
    w = w0.astype(float).copy()
    fixed = w <= 0                                   # 비중 0으로 묶인 자산 (working set)
    w[fixed] = 0.0
    for it in range(max_iter):
        F = np.flatnonzero(~fixed)                   # 비중이 자유로운 자산
        x = np.linalg.solve(S[np.ix_(F, F)], np.ones(len(F)))
        w_star = x / x.sum()                         # F 안에서의 GMV (등식 제약만)
        p = w_star - w[F]                            # 이동 방향

        if np.abs(p).max() < 1e-12:
            # 현재 집합에서 최적 → 묶인 자산을 풀어주면 더 좋아지는지(KKT) 확인
            lam = 1.0 / x.sum()                      # 라그랑주 승수 (= 포트폴리오 분산)
            mu = S @ w - lam                         # 묶인 자산의 승수 (≥ 0 이어야 최적)
            mu[~fixed] = np.inf
            j = np.argmin(mu)
            if mu[j] >= -tol:
                w = np.clip(w, 0, None)
                return w / w.sum(), it
            fixed[j] = False                         # 가장 크게 위반하는 자산을 풀어줌
        else:
            # p 방향으로 가되, 0 아래로 내려가는 자산이 생기면 그 지점에서 멈추고 묶음
            neg = p < -1e-15
            steps = np.full(len(F), np.inf)
            steps[neg] = -w[F][neg] / p[neg]
            k = np.argmin(steps)
            alpha = min(1.0, steps[k])
            w[F] += alpha * p
            if alpha < 1.0:
                fixed[F[k]] = True
                w[F[k]] = 0.0
    raise RuntimeError("GMV active-set 수렴 실패")

In [ ]:
# 검증: 무작위 날짜 하나에서 scipy SLSQP 결과와 비교 (같은 해가 나와야 함)
from scipy.optimize import minimize

i_test = cal.get_loc(cal[cal >= "2021-06-01"][0])
L = WINDOWS["6M"]
Rw = R[i_test - L + 1: i_test + 1]
ok = ~np.isnan(Rw).any(axis=0)
S_test, sh = estimate_cov(Rw[:, ok])
n = ok.sum()

t0 = time.time()
w_as, iters = gmv_long_only(S_test, np.full(n, 1 / n))
t_as = time.time() - t0

t0 = time.time()
res = minimize(lambda w: w @ S_test @ w, np.full(n, 1 / n), jac=lambda w: 2 * S_test @ w,
               bounds=[(0, 1)] * n, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
               method="SLSQP", options={"ftol": 1e-15, "maxiter": 1000})
t_sl = time.time() - t0

print(f"{cal[i_test].date()}, 자산 {n}개, 수축 강도 {sh:.2f}")
print(f"active-set: 분산 {w_as @ S_test @ w_as:.4e}, {iters}회, {t_as*1000:.1f}ms, 편입 {(w_as>1e-6).sum()}종목")
print(f"SLSQP     : 분산 {res.x @ S_test @ res.x:.4e}, {t_sl*1000:.1f}ms")
print(f"비중 최대 차이: {np.abs(w_as - res.x).max():.2e}")

## 4. 가상 포트폴리오: 매일 GMV 계산
- 날짜 t 장 마감 후, **t까지의 과거 L일 수익률**로 공분산을 추정해서 GMV 비중 $w_t$를 계산 → t+1 수익률에 적용 (미래 정보 사용 없음)
- 그날 투자 가능한 자산 = **윈도우 L일 동안 수익률이 하루도 빠짐없이 있는 자산** (회의록 "모든 종목이 있는 값만")
  → 1996년엔 코스피·나스닥만, 코인은 2017년 이후 상장 순서대로 자연스럽게 편입
- 표준편차가 0인 자산(윈도우 내내 가격이 같음 = 사실상 거래정지)은 제외

In [ ]:
def compute_gmv_weights(L):
    """롤링 윈도우 L일로 매일 GMV 비중 계산 → (거래일 × 자산) 배열, 일별 정보표"""
    T, N = R.shape
    W = np.zeros((T, N))
    info = []
    w_prev = np.zeros(N)
    t0 = time.time()
    for i in range(L, T):
        Rw = R[i - L + 1: i + 1]
        elig = ~np.isnan(Rw).any(axis=0)
        elig[elig] &= Rw[:, elig].std(axis=0) > 1e-10
        idx = np.flatnonzero(elig)
        if len(idx) < 2:
            continue
        S, shrink = estimate_cov(Rw[:, idx])

        # warm start: 전날 비중을 오늘 투자 가능한 자산으로 제한해서 재정규화
        w0 = w_prev[idx].copy()
        w0 = w0 / w0.sum() if w0.sum() > 0 else np.full(len(idx), 1 / len(idx))
        w, _ = gmv_long_only(S, w0)

        W[i, idx] = w
        w_prev = W[i]
        info.append((cal[i], len(idx), (w > 1e-6).sum(), shrink, np.sqrt(w @ S @ w * ANN)))
        if cal[i].year != cal[i - 1].year and cal[i].year % 5 == 0:
            print(f"  {cal[i].year} ... ({time.time() - t0:.0f}s)")
    info = pd.DataFrame(info, columns=["date", "n_eligible", "n_holdings", "shrink", "ex_ante_vol"]).set_index("date")
    return W, info

GMV_W, GMV_INFO = {}, {}
for wname, L in WINDOWS.items():
    t0 = time.time()
    GMV_W[wname], GMV_INFO[wname] = compute_gmv_weights(L)
    print(f"[{wname}] 완료: {time.time() - t0:.0f}s")

GMV_INFO["6M"].resample("YE").last().iloc[::3]

## 5. 실제 포트폴리오 시뮬레이션 (괴리율 리밸런싱)
매일 다음 순서로 진행:
1. **장중**: 실제 포트폴리오 보유금액이 각 자산 수익률만큼 변함 (매매 없음 → 비중이 흘러감). 가상 포트폴리오는 전날 GMV 비중으로 수익
2. **장 마감 후**: 마지막 리밸런싱 시점부터의 누적수익률로 괴리율 계산
   $\text{gap} = \left| \dfrac{V_t}{V_{\text{sync}}} - \dfrac{N_t}{N_{\text{sync}}} \right|$  ($V$: 가상, $N$: 실제 순자산)
3. gap ≥ 임계값이면 **신호**. `EXEC_LAG = 1`이면 **다음 거래일 종가**에, 신호 낸 날 계산한 GMV 비중으로 맞추고 비용 차감, 기준점 초기화
   - 신호 다음 날 하루 동안의 가격 변동은 그대로 맞음 (현실에서는 장 마감 후 계산 → 다음 날 주문)
   - 체결가 = 체결일 종가(원주가), 환율 = 체결일 환율
   - **휴장일 처리**: 체결일에 매매할 종목(보유 중이거나 목표 비중이 있는 종목)의 시장이 하나라도 쉬면 체결하지 않고 다음 날로 미룸.
     모든 시장이 열린 첫날, 신호 낸 날의 비중으로 체결 (미루는 동안 새 신호는 받지 않음)
     → 한 시장만 먼저 매매하면 통화·비중이 어긋난 채로 남기 때문에, 나눠서 체결하지 않고 한꺼번에 체결
   - **매매비용** = Σ |자산별 매매금액| × 시장별 편도비용
   - **환전비용** = 통화 주머니(KRW·USD·USDT) 사이에 실제로 옮겨야 하는 순금액 × 환전 스프레드
     (예: 코스피를 팔고 나스닥을 사면 그 차액만큼 원화→달러 환전. 같은 통화 안에서 사고파는 건 환전 없음)

- 투자 시작일: **원화 현금 1억**에서 (전날 계산한) GMV 비중대로 매수 (달러·USDT 자산 몫은 환전 후 매수) → 시작 시점 괴리율 0 (회의록 3번)
- **정수 주 제약**: 목표 금액 ÷ 그날 1주 가격(원주가)을 내림 → 남은 돈으로 '1주에 가장 가깝게 모자란' 종목부터 1주씩 추가
  - 목표 금액이 1주 가격보다 작은 종목(예: 비중 0.3% = 30만 원인데 1주 100만 원)은 못 삼
  - 못 쓴 돈은 원화 현금으로 보유 (수익 0) → 실제 포트폴리오 순자산 = 보유 주식 + 현금
- 보유 중 데이터가 끊긴 자산(상장폐지·거래정지): 수익률 0으로 두고 다음 리밸런싱 때 정리
- 임계값 0 = 매일 GMV로 리밸런싱(비용 포함, t+1이면 전날 비중으로 매일 체결) → 괴리율 규칙의 효과를 보는 비교 기준
- 임계값 ∞ = 첫 매수 후 그대로 보유

In [ ]:
def fx_cost(cur, tgt):
    """
    통화 주머니 간 환전비용
    cur, tgt: 통화별(KRW, USD, USDT) 현재·목표 금액 (원화 환산)
    3개 통화라서 '보내는 쪽'이나 '받는 쪽' 중 하나는 항상 1개 이하 → 송금 경로가 하나로 정해짐
    (원화↔USDT 1.05% = 원화↔달러 0.90% + 달러↔USDT 0.15% 이므로 직접 환전과 달러 경유 비용이 같음)
    """
    d = tgt - cur
    out = [(CCYS[k], -d[k]) for k in range(3) if d[k] < 0]    # 돈이 나가는 통화
    inn = [(CCYS[k], d[k]) for k in range(3) if d[k] > 0]     # 돈이 들어오는 통화
    if not out or not inn:
        return 0.0
    if len(out) == 1:
        src = out[0][0]
        return sum(amt * FX_COST[frozenset({src, dst})] for dst, amt in inn)
    dst = inn[0][0]
    return sum(amt * FX_COST[frozenset({src, dst})] for src, amt in out)

def rebalance_cost(h_cur, h_tgt, cash_cur=0.0, cash_tgt=0.0):
    """
    h_cur → h_tgt 로 바꿀 때의 (매매비용, 환전비용)  [금액, 원]
    cash_cur, cash_tgt: 매매 전·후 원화 현금 (현금은 KRW 주머니에 포함)
    """
    trade = cost_vec @ np.abs(h_tgt - h_cur)
    cur_ccy = ccy_onehot @ h_cur + np.array([cash_cur, 0, 0])
    tgt_ccy = ccy_onehot @ h_tgt + np.array([cash_tgt, 0, 0])
    fx = fx_cost(cur_ccy, tgt_ccy)
    return trade, fx

def round_to_shares(target, i):
    """
    목표 보유금액(원) → 실제로 살 수 있는 금액 (정수 주)
    1) 정수 주 자산: 내림해서 살 수 있는 만큼만
    2) 남은 돈으로 '다음 1주까지 가장 조금 모자란' 종목부터 1주씩 추가 (한 바퀴)
    코인(INT_MASK=False)은 소수점 매매 가능 → 목표 금액 그대로
    """
    out = target.copy()
    m = np.flatnonzero(INT_MASK & (target > 0))
    p = UNIT_PX[i, m]
    p = np.where(np.isfinite(p) & (p > 0), p, np.inf)        # 가격 없으면 매수 불가
    n = np.floor(target[m] / p)
    left = target.sum() - (n * p).sum() - out[~INT_MASK].sum()
    frac = target[m] / p - n                                   # 1주 중 모자란 비율 (0~1)
    for k in np.argsort(-frac):
        if frac[k] > 0 and p[k] <= left:
            n[k] += 1
            left -= p[k]
    out[m] = np.where(np.isfinite(p), n * p, 0.0)
    return out

def rebalance(h, cash, w, i):
    """
    날짜 i에 현재 보유(h, cash)를 비중 w로 리밸런싱 → (새 보유, 새 현금, 매매비용, 환전비용)
    비용을 먼저 대략 계산해서 남겨둔 뒤 나머지로 정수 주 매수
    """
    nav = h.sum() + cash
    tc, fc = rebalance_cost(h, w * nav, cash)                  # 1차 비용 추정 (소수점 매매 가정)
    budget = nav - (tc + fc) * 1.02                            # 비용 + 여유분(2%) 남기기
    h_new = round_to_shares(w * budget, i)
    tc, fc = rebalance_cost(h, h_new, cash, nav - h_new.sum() - tc - fc)   # 실제 매매 기준으로 재계산
    cash_new = nav - h_new.sum() - tc - fc
    return h_new, cash_new, tc, fc

def simulate(W, start, end, theta, integer=True, lag=None):
    """
    W: 가상 GMV 비중 배열 (row i = 날짜 i 장 마감 후 결정한 비중)
    start, end: 시뮬레이션 기간 / theta: 괴리율 임계값 (np.inf = 리밸런싱 없음)
    integer: True = 1억 + 정수 주 제약 / False = 소수점 매매 (비교용)
    lag: 체결 지연 (None이면 EXEC_LAG). 1 = 신호 다음 날 종가에 체결, 0 = 그날 종가에 체결
    반환: dict(nav=실제 순자산(시작=1), vnav=가상 순자산, h=마지막 보유금액, cash=마지막 현금, rebal=리밸런싱 기록)
    """
    lag = EXEC_LAG if lag is None else lag
    i0 = cal.searchsorted(start)                         # 시작일 (첫 매수일)
    i1 = cal.searchsorted(end, side="right") - 1         # 종료일

    def all_open(i, h, w):
        """날짜 i에 h → w 매매에 필요한 시장(보유 중이거나 목표 비중이 있는 종목의 시장)이 모두 열렸는가"""
        need = (h > 0) | (w > 1e-12)
        return ASSET_OPEN[i, need].all()

    # 첫 매수일이 휴장이면 필요한 시장이 모두 열린 날까지 미룸 (그 사이 날짜는 결과에서 빠짐)
    while not all_open(i0, np.zeros(len(assets)), W[i0 - lag]):
        i0 += 1
    R0 = np.nan_to_num(R)                                # 데이터 없는 날 수익률 = 0

    rb = rebalance if integer else rebalance_frac

    # 투자 시작: 원화 현금 1억 → (필요한 만큼 환전) → GMV 비중대로 매수
    #   lag=1이면 시작일 전날 장 마감에 계산한 비중을 시작일 종가에 체결
    h, cash, tc, fc = rb(np.zeros(len(assets)), CAPITAL, W[i0 - lag], i0)
    nav = h.sum() + cash
    v = 1.0                                              # 가상 포트폴리오 가치 (비용 없음, 비율)
    nav_sync, v_sync = nav, v                            # 괴리율 기준점

    navs, vnavs = [nav], [v]
    # (체결일, 신호일, 신호일 괴리율, 회전율, 매매비용%, 환전비용%, 현금비중, 편입종목수)
    log = [(cal[i0], cal[i0 - lag], 0.0, 1.0, tc / CAPITAL, fc / CAPITAL, cash / nav, (h > 0).sum(), 0)]
    pending = None                                       # (신호일 index, 신호일 괴리율) — 체결 대기

    def execute(i, i_sig, gap_sig):
        """날짜 i 종가에, 날짜 i_sig에 계산한 GMV 비중으로 리밸런싱하고 기준점 초기화"""
        nonlocal h, cash, nav, nav_sync, v_sync
        h_old, nav_old = h, nav
        h, cash, tc, fc = rb(h, cash, W[i_sig], i)
        log.append((cal[i], cal[i_sig], gap_sig, np.abs(h - h_old).sum() / 2 / nav_old,
                    tc / nav_old, fc / nav_old, cash / (h.sum() + cash), (h > 0).sum(),
                    i - i_sig - lag))                    # 휴장 때문에 미뤄진 거래일 수
        nav = h.sum() + cash
        nav_sync, v_sync = nav, v

    for i in range(i0 + 1, i1 + 1):
        r = R0[i]
        h = h * (1 + r)                                  # 실제: 보유 그대로 가격만 변동 (현금은 그대로)
        nav = h.sum() + cash
        v *= 1 + W[i - 1] @ r                            # 가상: 전날 GMV 비중으로 수익

        # (1) 대기 중인 신호: 체결일이 됐고 필요한 시장이 모두 열렸으면 체결 (신호일 비중으로)
        if pending is not None and i - pending[0] >= lag and all_open(i, h, W[pending[0]]):
            execute(i, *pending)
            pending = None

        # (2) 오늘 장 마감 후 괴리율 확인 → 신호 (체결 대기 중이면 새 신호 안 받음)
        if pending is None:
            gap = abs(v / v_sync - nav / nav_sync)       # 마지막 동기화 이후 누적수익률 차이
            if gap >= theta:
                if lag == 0 and all_open(i, h, W[i]):
                    execute(i, i, gap)                   # 당일 체결 (v1 방식, 시장이 열렸을 때만)
                else:
                    pending = (i, gap)                   # 다음 거래일 이후 체결
        navs.append(nav)
        vnavs.append(v)

    idx = cal[i0: i1 + 1]
    return dict(nav=pd.Series(navs, idx) / CAPITAL, vnav=pd.Series(vnavs, idx), h=h, cash=cash,
                rebal=pd.DataFrame(log, columns=["date", "signal_date", "gap", "turnover", "trade_cost",
                                                 "fx_cost", "cash_ratio", "n_holdings", "holiday_delay"]).set_index("date"))

def rebalance_frac(h, cash, w, i=None):
    """비교용: 소수점 매매가 가능하다고 가정한 리밸런싱 (현금 0)"""
    nav = h.sum() + cash
    tc, fc = rebalance_cost(h, w * nav, cash)
    return w * (nav - tc - fc), 0.0, tc, fc

## 6. 성과 지표
| 지표 | 정의 |
|---|---|
| CAGR | 연복리 수익률 (시작 원금 1 기준, 첫 매수 비용 포함) |
| 변동성 | 일간 수익률 표준편차 × √252 |
| 샤프 | 연환산 평균 수익률 ÷ 변동성 (무위험 수익률 = 현금 = 0) |
| 소르티노 | 연환산 평균 수익률 ÷ 하방 변동성 |
| MDD | 최대 낙폭 (고점 대비 최대 하락률) |
| 칼마 | CAGR ÷ \|MDD\| |
| 리밸런싱 / 연회전율 | 리밸런싱 횟수, 연평균 (매매금액/2 ÷ 순자산) |
| 연매매비용 / 연환전비용 | 연평균 비용 (순자산 대비, 첫 매수 포함) |
| 평균현금비중 | 리밸런싱 직후 현금 비중의 평균 (정수 주로 못 산 돈) |

In [ ]:
def perf(nav, rebal=None):
    """순자산 시계열(시작 원금 1)로 성과 지표 계산"""
    r = pd.concat([pd.Series([1.0]), nav.reset_index(drop=True)]).pct_change().dropna()
    years = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = nav.iloc[-1] ** (1 / years) - 1
    vol = r.std() * np.sqrt(ANN)
    downside = np.sqrt((np.minimum(r, 0) ** 2).mean()) * np.sqrt(ANN)
    mdd = (nav / nav.cummax().clip(lower=1.0) - 1).min()
    out = {
        "CAGR": cagr, "변동성": vol,
        "샤프": r.mean() * ANN / vol, "소르티노": r.mean() * ANN / downside,
        "MDD": mdd, "칼마": cagr / abs(mdd),
        "누적수익률": nav.iloc[-1] - 1,
    }
    if rebal is not None:
        out["리밸런싱"] = len(rebal) - 1                    # 첫 매수 제외
        out["연회전율"] = rebal["turnover"].iloc[1:].sum() / years
        out["연매매비용"] = rebal["trade_cost"].sum() / years
        out["연환전비용"] = rebal["fx_cost"].sum() / years
        out["평균현금비중"] = rebal["cash_ratio"].mean()
        out["휴장지연"] = int((rebal["holiday_delay"].iloc[1:] > 0).sum())   # 휴장 때문에 미뤄진 리밸런싱 횟수
    return out

RANK_COLS = ["CAGR", "샤프", "소르티노", "칼마", "MDD"]     # 종합 순위에 쓰는 지표 (모두 클수록 좋음)

def run_all(start, end, lag=None):
    """32개 시나리오(윈도우 2 × 괴리율 16) + 비교 기준(가상 GMV, 매일 리밸런싱) 실행 → 결과표, 시뮬레이션 결과"""
    rows, sims = [], {}
    for wname in WINDOWS:
        for theta in [0.0] + THRESHOLDS:
            s = simulate(GMV_W[wname], start, end, theta, lag=lag)
            sims[(wname, theta)] = s
            rows.append({"윈도우": wname, "괴리율": theta, **perf(s["nav"], s["rebal"])})
        # 가상 GMV 자체 (거래비용 없음 = 이론상 상한)
        rows.append({"윈도우": wname, "괴리율": np.nan, **perf(s["vnav"])})
    res = pd.DataFrame(rows)
    res["구분"] = np.where(res["괴리율"].isna(), "가상 GMV(비용 없음)",
                   np.where(res["괴리율"] == 0, "매일 리밸런싱(비용 포함)", "시나리오"))
    return res, sims

def rank_scenarios(res):
    """시나리오 32개(리밸런싱 없음 포함)만 골라 5개 지표 순위의 평균으로 종합 순위"""
    sc = res[res["구분"] == "시나리오"].copy()
    for c in RANK_COLS:
        sc[f"{c}_순위"] = sc[c].rank(ascending=False, method="min")
    sc["평균순위"] = sc[[f"{c}_순위" for c in RANK_COLS]].mean(axis=1)
    return sc.sort_values(["평균순위", "샤프"], ascending=[True, False])

## 7. In-sample 백테스트 (~ 2024-12-31)
두 윈도우가 같은 날 시작하도록, 6개월 윈도우 GMV가 처음 나온 **다음 날**(t+1 체결)을 공통 시작일로 사용

In [ ]:
IS_START = cal[max(WINDOWS.values()) + EXEC_LAG]
t0 = time.time()
res_is, sims_is = run_all(IS_START, IS_END)
print(f"In-sample: {IS_START.date()} ~ {IS_END.date()} ({time.time() - t0:.0f}s)")

ranked_is = rank_scenarios(res_is)
BEST = (ranked_is.iloc[0]["윈도우"], ranked_is.iloc[0]["괴리율"])
print(f"▶ 최적 조합: 윈도우 {BEST[0]}, 괴리율 {tlabel(BEST[1])}")

fmt = {"CAGR": "{:.2%}", "변동성": "{:.2%}", "샤프": "{:.3f}", "소르티노": "{:.3f}", "MDD": "{:.2%}",
       "칼마": "{:.3f}", "누적수익률": "{:.1%}", "연회전율": "{:.2f}", "연매매비용": "{:.3%}", "연환전비용": "{:.3%}", "평균현금비중": "{:.2%}", "괴리율": tlabel,
       "평균순위": "{:.1f}", "리밸런싱": "{:.0f}", "휴장지연": "{:.0f}"}
show_cols = ["윈도우", "괴리율", "CAGR", "변동성", "샤프", "소르티노", "MDD", "칼마", "리밸런싱", "휴장지연", "연회전율", "연매매비용", "연환전비용", "평균현금비중", "평균순위"]
ranked_is[show_cols].style.format(fmt, na_rep="-").hide(axis="index")

In [ ]:
# 비교 기준: 가상 GMV(비용 없음) / 매일 리밸런싱(비용 포함) vs 최적 시나리오
bench_is = res_is[res_is["구분"] != "시나리오"]
pd.concat([bench_is, ranked_is.head(1)])[["구분"] + show_cols[:-1]].style.format(fmt, na_rep="-").hide(axis="index")

In [ ]:
# 차트 1: 시나리오별 샤프·칼마·MDD 히트맵 (윈도우 × 괴리율)
fig, axes = plt.subplots(3, 1, figsize=(13, 6.6), constrained_layout=True)
sc = res_is[res_is["구분"] == "시나리오"]
for ax, (col, f) in zip(axes, [("샤프", "{:.2f}"), ("칼마", "{:.2f}"), ("MDD", "{:.1%}")]):
    mat = sc.pivot(index="윈도우", columns="괴리율", values=col).loc[list(WINDOWS)]
    im = ax.imshow(mat.values, cmap=SEQ_BLUE, aspect="auto")   # 진할수록 좋음 (MDD도 덜 빠질수록 큰 값)
    ax.set_xticks(range(mat.shape[1]), [tlabel(t) for t in mat.columns])
    ax.set_yticks(range(mat.shape[0]), mat.index)
    ax.grid(False)
    lo, hi = np.nanmin(mat.values), np.nanmax(mat.values)
    for (r_, c_), val in np.ndenumerate(mat.values):
        dark = (val - lo) / (hi - lo + 1e-12) > 0.55
        ax.text(c_, r_, f.format(val), ha="center", va="center", fontsize=8,
                color="white" if dark else "#0b0b0b")
    ax.set_title(f"{col} (진할수록 좋음)", loc="left", fontsize=11)
    if col == "MDD":
        ax.set_xlabel("괴리율 임계값")
    # 최적 조합 표시
    bi, bj = list(WINDOWS).index(BEST[0]), list(mat.columns).index(BEST[1])
    ax.add_patch(plt.Rectangle((bj - .5, bi - .5), 1, 1, fill=False, ec="#eb6834", lw=2))
fig.suptitle(f"In-sample ({IS_START.date()} ~ {IS_END.date()}) 시나리오 성과 — 주황 테두리 = 종합 최적",
             x=0.0, ha="left", fontsize=12)
fig.savefig(OUT_DIR / "fig1_heatmap_insample.png")
plt.show()

In [ ]:
def plot_nav(sims, best, title, fname):
    """최적 조합 vs 가상 GMV vs 매일 리밸런싱 순자산 비교 (로그 스케일)"""
    s_best = sims[best]
    s_daily = sims[(best[0], 0.0)]
    series = [
        (f"최적: {best[0]} · 괴리율 {tlabel(best[1])}", s_best["nav"], C_BLUE),
        ("가상 GMV (비용 없음)", s_best["vnav"], C_ORANGE),
        ("매일 리밸런싱 (비용 포함, t+1)", s_daily["nav"], C_AQUA),
    ]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for label, s, color in series:
        ax.plot(s.index, s.values, color=color, lw=1.6, label=label)
        ax.annotate(f"{s.iloc[-1]:.2f}", (s.index[-1], s.iloc[-1]), xytext=(4, 0),
                    textcoords="offset points", va="center", fontsize=9, color=TEXT_2)
    # 리밸런싱 시점 표시 (최적 조합)
    rb = s_best["rebal"].index[1:]
    ax.plot(rb, s_best["nav"].reindex(rb), "|", color=C_BLUE, ms=7, alpha=0.5, label="리밸런싱 시점")
    lo = min(s.min() for _, s, _ in series); hi = max(s.max() for _, s, _ in series)
    if hi / lo > 3:                                  # 변동 폭이 클 때만 로그 스케일
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:g}"))
        ax.yaxis.set_minor_formatter(plt.NullFormatter())
        ax.set_ylabel("순자산 (시작 = 1, 로그)")
    else:
        ax.set_ylabel("순자산 (시작 = 1)")
    ax.set_title(title, loc="left", fontsize=12)
    ax.legend(frameon=False, loc="upper left", fontsize=9)
    fig.savefig(OUT_DIR / fname)
    plt.show()

plot_nav(sims_is, BEST, f"In-sample 순자산 ({IS_START.date()} ~ {IS_END.date()})", "fig2_nav_insample.png")

## 8. Out-of-sample (2025-01-01 ~ 최신)
- 선택한 조합으로 **2025년 첫 거래일에 새로 투자 시작** (가상·실제 모두 1에서 출발)
- 첫날 GMV는 2024년 하반기 데이터로 계산 → 미래 정보 없음
- 나머지 31개 시나리오도 참고용으로 같이 계산 (**선택에는 쓰지 않음**)

In [ ]:
res_oos, sims_oos = run_all(OOS_START, END)
oos_start = sims_oos[BEST]["nav"].index[0]
print(f"Out-of-sample: {oos_start.date()} ~ {END.date()}")

best_oos = res_oos[(res_oos["윈도우"] == BEST[0]) & (res_oos["괴리율"] == BEST[1])]
bench_oos = res_oos[(res_oos["윈도우"] == BEST[0]) & (res_oos["구분"] != "시나리오")]
summary_oos = pd.concat([best_oos.assign(구분="선택 조합"), bench_oos])
summary_oos[["구분"] + show_cols[:-1]].style.format(fmt, na_rep="-").hide(axis="index")

In [ ]:
# 참고: OOS에서의 32개 시나리오 순위 (in-sample 최적이 OOS에서 몇 등인지 확인)
ranked_oos = rank_scenarios(res_oos)
ranked_oos["IS순위"] = ranked_oos.apply(
    lambda r: ranked_is.reset_index(drop=True).index[
        (ranked_is["윈도우"].values == r["윈도우"]) & (ranked_is["괴리율"].values == r["괴리율"])][0] + 1, axis=1)
ranked_oos[show_cols + ["IS순위"]].style.format(fmt, na_rep="-").hide(axis="index")

In [ ]:
plot_nav(sims_oos, BEST, f"Out-of-sample 순자산 ({oos_start.date()} ~ {END.date()})", "fig3_nav_oos.png")

## 9. 가상 GMV는 무엇을 들고 있었나

In [ ]:
# 차트 4: 시장별 비중 추이 (선택 윈도우, 월말 기준)
Wb = pd.DataFrame(GMV_W[BEST[0]], index=cal, columns=assets)
by_mkt = Wb.T.groupby(meta["market"]).sum().T.loc[IS_START:].resample("ME").last()
fig, ax = plt.subplots(figsize=(11, 3.8))
colors = {"KOSPI": C_BLUE, "NASDAQ": C_ORANGE, "CRYPTO": C_AQUA}
ax.stackplot(by_mkt.index, [by_mkt[m] for m in colors], colors=list(colors.values()),
             labels=list(colors), edgecolor="white", linewidth=0.5)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.axvline(OOS_START, color=TEXT_2, lw=1, ls="--")
ax.text(OOS_START, 1.01, " OOS", fontsize=9, color=TEXT_2, va="bottom")
ax.set_title(f"가상 GMV({BEST[0]}) 시장별 비중 (월말)", loc="left", fontsize=12)
ax.legend(frameon=False, ncol=3, loc="upper left", bbox_to_anchor=(0, -0.08))
fig.savefig(OUT_DIR / "fig4_gmv_market_weights.png")
plt.show()

In [ ]:
# 기간 평균 비중 상위 15종목 (in-sample / out-of-sample)
def top_holdings(W_df, start, end, k=15):
    avg = W_df.loc[start:end].mean().sort_values(ascending=False).head(k)
    return pd.DataFrame({"이름": meta.loc[avg.index, "name"], "시장": meta.loc[avg.index, "market"],
                         "평균비중": avg.map("{:.1%}".format)})

pd.concat({"In-sample": top_holdings(Wb, IS_START, IS_END).reset_index(),
           "Out-of-sample": top_holdings(Wb, OOS_START, END).reset_index()}, axis=1)

## 10. 참고: OOS 시작일의 효율적 프런티어
회의록 2번 "포트폴리오 구성은 효율적 프런티어"의 그림. GMV는 프런티어의 가장 왼쪽 점
(롱온리, 선택 윈도우의 평균 수익률·공분산 사용, 연율화)

In [ ]:
i_f = cal.searchsorted(OOS_START) - 1                    # OOS 직전 거래일 (2024년 마지막 거래일)
L = WINDOWS[BEST[0]]
Rw = R[i_f - L + 1: i_f + 1]
elig = ~np.isnan(Rw).any(axis=0)
elig[elig] &= Rw[:, elig].std(axis=0) > 1e-10
idx = np.flatnonzero(elig)
S_f = estimate_cov(Rw[:, idx])[0] * ANN
mu_f = Rw[:, idx].mean(axis=0) * ANN
w_gmv = GMV_W[BEST[0]][i_f, idx]

# 목표 수익률별 최소분산 포트폴리오 (SLSQP, 롱온리)
targets = np.linspace(mu_f @ w_gmv, mu_f.max() * 0.98, 15)
front, w_init = [], w_gmv.copy()
for tgt in targets:
    r_ = minimize(lambda w: w @ S_f @ w, w_init, jac=lambda w: 2 * S_f @ w, method="SLSQP",
                  bounds=[(0, 1)] * len(idx),
                  constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1},
                               {"type": "ineq", "fun": lambda w, t=tgt: mu_f @ w - t}],
                  options={"maxiter": 500, "ftol": 1e-12})
    if r_.success:
        front.append((np.sqrt(r_.x @ S_f @ r_.x), mu_f @ r_.x))
        w_init = r_.x
front = np.array(front)

fig, ax = plt.subplots(figsize=(9, 5))
mk = meta.loc[assets[idx], "market"].to_numpy()
for m, c in colors.items():
    sel = mk == m
    ax.scatter(np.sqrt(np.diag(S_f))[sel], mu_f[sel], s=14, color=c, alpha=0.55, label=f"{m} 개별 자산",
               edgecolor="white", linewidth=0.5)
ax.plot(front[:, 0], front[:, 1], color="#0b0b0b", lw=1.8, label="효율적 프런티어 (롱온리)")
ax.scatter([np.sqrt(w_gmv @ S_f @ w_gmv)], [mu_f @ w_gmv], s=70, color="#0b0b0b", zorder=5)
ax.annotate("GMV", (np.sqrt(w_gmv @ S_f @ w_gmv), mu_f @ w_gmv), xytext=(8, -4),
            textcoords="offset points", fontsize=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_xlabel("연 변동성"); ax.set_ylabel("연 기대수익률 (과거 평균)")
ax.set_title(f"{cal[i_f].date()} 기준 효율적 프런티어 ({BEST[0]} 윈도우, {len(idx)}개 자산)", loc="left", fontsize=12)
ax.legend(frameon=False, fontsize=9)
fig.savefig(OUT_DIR / "fig5_frontier.png")
plt.show()

## 11. 1억 정수 주 제약의 영향
같은 조합을 "소수점 매매 가능"으로 돌린 결과와 비교 → 1억이라는 규모 때문에 생기는 손실(현금 보유, 못 산 종목) 확인

In [ ]:
cmp_rows = []
for tag, (st, en) in {"In-sample": (IS_START, IS_END), "Out-of-sample": (OOS_START, END)}.items():
    for integer, label in [(True, "1억 · 정수 주"), (False, "소수점 매매 (이상적)")]:
        s_ = simulate(GMV_W[BEST[0]], st, en, BEST[1], integer=integer)
        cmp_rows.append({"기간": tag, "방식": label, **perf(s_["nav"], s_["rebal"]),
                         "평균편입종목": s_["rebal"]["n_holdings"].mean()})
cmp = pd.DataFrame(cmp_rows)
cmp[["기간", "방식", "CAGR", "변동성", "샤프", "MDD", "칼마", "리밸런싱", "연매매비용", "연환전비용",
     "평균현금비중", "평균편입종목"]].style.format({**fmt, "평균편입종목": "{:.1f}"}).hide(axis="index")

## 12. 지금 1억으로 시작한다면: 주문표
- 데이터 마지막 날(`END`) 장 마감에 계산한 GMV 비중을 **다음 거래일에 주문**한다고 보고, 선택한 윈도우의 GMV 비중을 **1억 원 · 정수 주**로 바꾼 매수 목록
- 가격은 그날 종가(원주가), 달러 자산은 그날 환율로 원화 환산. 실제 주문 시점의 가격·환율과는 차이가 있음
- 코인은 소수점 수량

In [ ]:
i_last = len(cal) - 1
w_now = GMV_W[BEST[0]][i_last]
h_now, cash_now, tc_now, fc_now = rebalance(np.zeros(len(assets)), CAPITAL, w_now, i_last)

held = np.flatnonzero((w_now > 1e-6) | (h_now > 0))
fx_last = fx_cal.iloc[i_last] if BASE_CCY == "KRW" else 1.0
orders = pd.DataFrame({
    "종목": meta["name"].to_numpy()[held],
    "시장": meta["market"].to_numpy()[held],
    "티커": [a.split(":")[1] for a in assets[held]],
    "현지가격": px_raw.iloc[i_last].to_numpy()[held] / np.where(meta["ccy"].to_numpy()[held] == "USD", fx_last, 1.0),
    "수량": h_now[held] / UNIT_PX[i_last, held],
    "금액(원)": h_now[held],
    "목표비중": w_now[held],
    "실제비중": h_now[held] / CAPITAL,
}).sort_values("목표비중", ascending=False)
orders["못삼"] = orders["수량"] == 0

print(f"기준일 {cal[i_last].date()}, 윈도우 {BEST[0]}, USD/KRW {fx_last:,.2f}")
print(f"GMV 편입 {(w_now > 1e-6).sum()}종목 → 실제 매수 {(h_now > 0).sum()}종목 "
      f"(목표 금액이 1주 가격보다 작아 못 산 종목 {orders['못삼'].sum()}개, 목표비중 합 {orders.loc[orders['못삼'], '목표비중'].sum():.2%})")
print(f"주식·코인 {h_now.sum():,.0f}원 + 현금 {cash_now:,.0f}원 + 매매비용 {tc_now:,.0f}원 + 환전비용 {fc_now:,.0f}원 = {h_now.sum() + cash_now + tc_now + fc_now:,.0f}원")
orders.style.format({"현지가격": "{:,.2f}", "수량": "{:,.4g}", "금액(원)": "{:,.0f}",
                     "목표비중": "{:.2%}", "실제비중": "{:.2%}"}).hide(axis="index")

## 13. t+1 체결의 영향 / v1 최적 조합과 비교
같은 32개 시나리오를 **그날 종가 체결(lag = 0, v1 방식)** 로도 돌려서 체결 지연만의 효과를 확인
- 차이가 크면 v1 성과의 일부는 "괴리율을 본 그 가격에 바로 거래할 수 있다"는 비현실적 가정 덕분
- v1 최적 조합(3M · 9%)이 v2 조건에서 어떻게 되는지도 같이 확인

In [ ]:
t0 = time.time()
res_is0, sims_is0 = run_all(IS_START, IS_END, lag=0)
res_oos0, sims_oos0 = run_all(OOS_START, END, lag=0)
print(f"lag=0 재계산 ({time.time() - t0:.0f}s)")

key = ["윈도우", "괴리율"]
cols_cmp = ["CAGR", "샤프", "MDD", "칼마", "리밸런싱"]
lag_cmp = (res_is[res_is["구분"] != "가상 GMV(비용 없음)"].set_index(key)[cols_cmp]
           .join(res_is0[res_is0["구분"] != "가상 GMV(비용 없음)"].set_index(key)[cols_cmp], lsuffix="_t+1", rsuffix="_당일"))
for c in ["CAGR", "샤프"]:
    lag_cmp[f"{c}_차이"] = lag_cmp[f"{c}_t+1"] - lag_cmp[f"{c}_당일"]
lag_cmp = lag_cmp.reset_index()
lag_cmp.style.format({**{f"{c}_{t}": fmt[c] for c in cols_cmp for t in ["t+1", "당일"]},
                      "CAGR_차이": "{:+.2%}", "샤프_차이": "{:+.3f}", "괴리율": tlabel}).hide(axis="index")

In [ ]:
# 차트 6: 괴리율별 샤프 (윈도우 × 체결 시점) — 괴리율은 순서가 있는 범주라 선 그래프
sc1 = res_is[res_is["구분"] == "시나리오"]
sc0 = res_is0[res_is0["구분"] == "시나리오"]
xs = np.arange(len(THRESHOLDS))
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, col in zip(axes, ["샤프", "칼마"]):
    for wname, c in zip(WINDOWS, [C_BLUE, C_ORANGE]):
        for df_, ls, tag in [(sc1, "-", "t+1"), (sc0, "--", "당일")]:
            y = df_[df_["윈도우"] == wname].set_index("괴리율").loc[THRESHOLDS, col]
            ax.plot(xs, y.values, ls=ls, color=c, lw=2, marker="o", ms=4, label=f"{wname} · {tag} 체결")
    ax.set_xticks(xs, [tlabel(t) for t in THRESHOLDS], fontsize=8)
    ax.set_xlabel("괴리율 임계값")
    ax.set_title(f"In-sample {col}", loc="left", fontsize=12)
    if BEST[1] in THRESHOLDS:
        ax.axvline(THRESHOLDS.index(BEST[1]), color=TEXT_2, lw=1, ls=":")
axes[0].legend(frameon=False, fontsize=9)
fig.savefig(OUT_DIR / "fig6_lag_effect.png")
plt.show()

In [ ]:
# v1 최적(3M · 9%) vs v2 최적: 체결 시점별, IS / OOS
V1_BEST = ("3M", 0.09)
rows = []
for name, combo in [("v1 최적 (3M · 9%)", V1_BEST), (f"v2 최적 ({BEST[0]} · {tlabel(BEST[1])})", BEST)]:
    for lag_tag, r_is, r_oos in [("당일", res_is0, res_oos0), ("t+1", res_is, res_oos)]:
        for per, r_ in [("In-sample", r_is), ("Out-of-sample", r_oos)]:
            row = r_[(r_["윈도우"] == combo[0]) & (r_["괴리율"] == combo[1])].iloc[0]
            rows.append({"조합": name, "체결": lag_tag, "기간": per, **row[show_cols[2:-1]].to_dict()})
v1v2 = pd.DataFrame(rows)
v1v2.style.format(fmt, na_rep="-").hide(axis="index")

## 14. 저장

In [ ]:
res_is.to_csv(OUT_DIR / "results_insample_all.csv", index=False, encoding="utf-8-sig")
ranked_is.to_csv(OUT_DIR / "results_insample_ranked.csv", index=False, encoding="utf-8-sig")
res_oos.to_csv(OUT_DIR / "results_oos_all.csv", index=False, encoding="utf-8-sig")

# 선택 조합의 순자산·리밸런싱 기록
for tag, sims in [("insample", sims_is), ("oos", sims_oos)]:
    s = sims[BEST]
    pd.DataFrame({"real_nav": s["nav"], "virtual_gmv_nav": s["vnav"],
                  "daily_rebal_nav": sims[(BEST[0], 0.0)]["nav"]}).to_csv(OUT_DIR / f"nav_best_{tag}.csv")
    s["rebal"].to_csv(OUT_DIR / f"rebalance_log_best_{tag}.csv")

# 가상 GMV 비중 (0이 아닌 것만, long 형식)
for wname, W in GMV_W.items():
    Wdf = pd.DataFrame(W, index=cal, columns=assets)
    long = Wdf[Wdf.index >= IS_START].stack()
    long = long[long > 1e-6].rename("weight").reset_index()
    long.columns = ["date", "asset", "weight"]
    long["name"] = long["asset"].map(meta["name"])
    long.to_csv(OUT_DIR / f"gmv_weights_{wname}.csv", index=False, encoding="utf-8-sig")
    GMV_INFO[wname].to_csv(OUT_DIR / f"gmv_info_{wname}.csv")

cmp.to_csv(OUT_DIR / "integer_vs_fractional.csv", index=False, encoding="utf-8-sig")
lag_cmp.to_csv(OUT_DIR / "lag_effect_insample.csv", index=False, encoding="utf-8-sig")
res_is0.to_csv(OUT_DIR / "results_insample_all_lag0.csv", index=False, encoding="utf-8-sig")
res_oos0.to_csv(OUT_DIR / "results_oos_all_lag0.csv", index=False, encoding="utf-8-sig")
v1v2.to_csv(OUT_DIR / "v1_vs_v2_best.csv", index=False, encoding="utf-8-sig")
orders.to_csv(OUT_DIR / f"orders_1억_{cal[i_last].date()}.csv", index=False, encoding="utf-8-sig")
print("saved to", OUT_DIR)
sorted(p.name for p in OUT_DIR.iterdir())